# Лабораторна 5. Маршрутизація звернень університетської IT-підтримки

Англомовна служба підтримки отримує короткий опис проблеми і передає його одній із чотирьох команд. На вхід надходить текст, на виході маємо одну мітку `account`, `connectivity`, `learning_platform` або `printing`.

Межі класів відповідають тому, що треба полагодити. Неможливість увійти одночасно в Moodle і пошту — проблема облікового запису. Якщо вхід успішний і помилка тільки в курсі, звернення йде команді платформи. Мережевий принтер із доступним інтернетом належить команді друку. Фіксована схема робить прогноз придатним для автоматичної маршрутизації.

Промпт до генеративної моделі можна використати як прототип. Для цього навантаження невеликий класифікатор повертає мітку одним прямим проходом і вчиться межам категорій з розмічених прикладів. RAG корисний для пошуку інструкції з ремонту, тоді як тут потрібно розпізнати категорію в самому повідомленні. Якість перевіряється на окремому тестовому наборі.

In [ ]:
%pip install -q "transformers==4.57.3"

## Модель і дані

Використовую попередньо навчений англомовний [Google Small BERT](https://huggingface.co/google/bert_uncased_L-2_H-128_A-2), Apache 2.0. Це Base-енкодер із 2 шарами, прихованою розмірністю 128 і 2 головами уваги. Він підходить для коротких текстів і повного оновлення ваг на CPU або T4. Додаю нову голову класифікації на чотири класи.

Джерело даних — власний синтетичний корпус вигаданих звернень, створений для цієї лабораторної. Ліцензія прикладів — [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/). Формат — текст, мітка і поділ. 168 різних звернень, із них 112 для тренування, 24 для валідації та 32 для тесту. Тест містить окремі формулювання й складні випадки з кількома можливими ключовими словами. Валідаційний набір використовується для вибору епохи, тестовий — для фінального порівняння.

In [1]:
DATA = {
    "account": {
        "train": """
I forgot my university password and need a reset link.
My campus ID is locked after several bad password attempts.
How do I enrol a new authenticator app for my university account?
I lost my phone and cannot approve the login notification.
The password reset link expired before I could open it.
I changed my surname and my campus email username is wrong.
The account activation code for my student portal never arrived.
I get invalid credentials when I try to sign into my campus mailbox.
Please remove my old phone number from my account recovery settings.
Can you unlock my university account before my exam?
I received a university login alert that I didn't initiate.
The temporary password given at orientation isn't accepted.
My campus username stopped working after I re-enrolled.
The authenticator is generating codes that my sign-in page rejects.
How can a new teaching assistant activate their staff identity?
I can no longer access the backup email used for password recovery.
The single sign-on page keeps asking me to verify my identity.
I set up multi-factor authentication but left my security key at home.
My previous campus account has the wrong student number attached.
The reset form says there is no account matching my university email.
I need to change the recovery questions on my campus identity.
My university password was reset without my permission.
My student credentials expired during the semester break.
My browser remembers my password but I need to update it.
The verification email is being sent to an address I no longer own.
I have two campus identities and need help merging my login details.
The account setup page will not accept my activation token.
Signing in says my campus identity has been suspended.
""",
        "validation": """
I deleted the university authenticator profile by mistake.
My first-time sign-up invitation no longer works.
I know my username but the password recovery form rejects it.
The login is blocked until I replace a lost security key.
My verification code arrives after it has already timed out.
A recent password change locked me out of all university services.
""",
        "test": """
The portal asks for a second factor that I cannot access.
My old campus ID won't let me sign in after I returned from leave.
Can you restore access to my university mailbox after too many guesses?
The reset wizard refuses to recognise my student email.
I used my last backup code and need to add a new sign-in method.
The sign-in page says my identity is disabled even though I am enrolled.
My credentials fail on both Moodle and email before either service opens.
I just joined the university and have no activation message.
""",
    },
    "connectivity": {
        "train": """
Eduroam keeps disconnecting whenever I sit in the library.
My laptop cannot find the campus wireless network.
The Wi-Fi in the chemistry lecture hall has no internet access.
The Ethernet socket in my dorm room appears to be dead.
My computer connects to Wi-Fi but no websites will load.
Our seminar room's wireless signal drops during video calls.
The university VPN tunnel disconnects every few minutes.
I can browse locally but the campus VPN never completes its handshake.
The guest wireless network is very slow this afternoon.
My laptop reports no IP address on the residence network.
Is there an outage on the network in the engineering building?
The network icon says connected without internet after the router reboot.
The cable works at home but the lab wall socket gives no connection.
My device can't join Eduroam with the configuration profile installed.
The campus Wi-Fi does not appear on my tablet but other networks do.
I get dropped from the university wireless whenever I move floors.
Traffic on the VPN works for a minute and then stops.
The dorm network gives me a self-assigned IP instead of internet access.
Is the wireless access point on the second floor offline?
Campus Ethernet stopped responding after the power cut.
The library network has high latency while off-campus internet is fine.
My phone sees Eduroam but cannot connect to it.
I moved desks and the wired network connection is missing.
The campus guest network is stuck on its welcome page.
The university network is down in every room on this floor.
The VPN client connects but cannot reach any on-campus hosts.
The residence wireless switches on and off all evening.
My device gets an IP address but cannot reach the internet from campus.
""",
        "validation": """
My wireless session drops every time I cross the student centre.
None of our devices can connect in the basement classroom.
My wired campus connection cannot get a lease from DHCP.
The internet connection at the lab desktops disappears after lunch.
The guest network rejects new devices since the access point restart.
The VPN says connected but campus websites are unreachable.
""",
        "test": """
Wireless works in the courtyard but drops immediately inside the auditorium.
None of the Ethernet ports in the study room show a link light.
I can open my course from home, but nothing loads on campus Wi-Fi.
The VPN handshake succeeds and then the tunnel closes right away.
Video conferences stall only when I use the residence internet.
My phone and laptop both report no internet on Eduroam tonight.
What is the cause of today's campus network outage in the art building?
The dorm connection disappears whenever a second device joins.
""",
    },
    "learning_platform": {
        "train": """
I submitted the assignment in Moodle but it still shows as missing.
A course disappeared from my learning dashboard this morning.
The lecture recording will not play from the course page.
I need to replace an attachment that I submitted to Canvas.
My quiz closed before I could submit the last answer.
Moodle says I am not enrolled in the course I selected this term.
Where can I find the comments on my uploaded coursework?
The gradebook shows a blank row even though my essay was marked.
The link to week three resources on the course page is broken.
The Canvas calendar shows the wrong coursework deadline.
My instructor reopened the quiz but I cannot find another attempt.
I cannot open lecture slides attached to the learning module.
The course forum won't accept my reply to the discussion.
My assignment upload was rejected although it is a PDF.
How do I view the feedback rubric on my assignment?
The LMS page shows last semester's modules instead of the new ones.
My group presentation is missing from the online gradebook.
Can I resubmit the assignment even though the first file is visible?
The course announcement is missing for students in my section.
I can sign into Canvas but the course has no visible content.
I need access to last week's practice quiz on Moodle.
The learning page says my submission is still a draft.
My peer review task is not assigned in the course platform.
The module for today's lab is still hidden from my class.
I am seeing duplicate assignment entries in the learning platform.
The video embedded in the lesson shows permission denied.
The instructor published my grade but I cannot see the feedback.
The course download button gives an error for every lecture PDF.
""",
        "validation": """
My assessment file appears uploaded but the course says it is a draft.
An instructor published the weekly module but my group cannot see it.
I can open Moodle, but the seminar's feedback rubric is hidden.
A recorded workshop is missing from the course media list.
The quiz marks have not appeared on the learning dashboard.
A read-only warning stops me from posting in my course discussion.
""",
        "test": """
A course invitation accepted yesterday still hasn't added the class to my dashboard.
The learning platform accepted my essay but won't show a receipt for it.
How can I see the lecturer's annotation on my graded submission?
I opened the lesson and found that its video is marked private.
The test timer paused, but the platform counted the answer as late.
I can sign in correctly and use email, yet today's lab is hidden in Canvas.
The moderator can see my class post, but the discussion board hides it from me.
The course timetable is fine but the assignment link sends me to the wrong module.
""",
    },
    "printing": {
        "train": """
The library printer says there is a paper jam but I cannot see any paper.
My print job is held in the queue long after I tapped my card.
How do I top up the credit used by the campus printers?
The copier charged me twice for a single page.
Can you clear an old print job stuck on the shared machine?
All the pages from the lab printer came out blank.
The print station is out of toner and still accepts jobs.
My document was supposed to print in colour and came out grey.
The print portal deducted credit although the job failed.
I can't find the office printer in the list of available queues.
My student card won't release a print job at the printer.
The scanner on the campus multifunction printer is not sending files.
The copier keeps pulling multiple sheets through at once.
The campus print application says my balance is too low.
Double-sided pages from the shared printer are upside down.
The printer reported a jam and my document was never released.
I need to cancel a queued print before my balance is deducted.
My print job shows completed but nothing came out of the printer.
I changed paper size but the campus printer ignored the setting.
The printer in the library has an empty paper tray.
The copy machine refuses to scan to my university email.
I sent a lab report to print and the text is cut off at the margins.
The university printer says I have no permission to print in colour.
I paid for a print job and received someone else's page.
The printer driver installed but none of my jobs reach its queue.
My printer balance disappeared after a failed duplex job.
The shared copier is leaving a black stripe on every printed page.
The campus print kiosk will not accept my card to release pages.
""",
        "validation": """
Printing in the seminar building stalls before the pages are released.
My credit decreased but the library copy machine produced no sheets.
An empty toner warning appears when I send a colour print job.
My document comes out at the wrong paper size on the shared copier.
The release station lists someone else's print jobs under my card.
The scanner attached to the printer cannot send my file.
""",
        "test": """
The print log shows the file as complete, but the output tray is empty.
The copy machine fed through paper and printed the same page four times.
I can reach the internet, but the wireless printer won't accept any jobs.
A printer asks for my card but never releases the queued assignment.
After a paper jam, my account lost credit even though no pages came out.
The page has a grey band down the middle whenever I use the shared copier.
The library print station says I must choose a different output tray.
Only the campus scanner fails to deliver files to my working mailbox.
""",
    },
}

In [2]:
import os
from collections import Counter
from time import perf_counter

import torch
from IPython.display import Markdown, display
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer

SEED = 42
torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
hardware = torch.cuda.get_device_name() if device.type == "cuda" else "CPU"
print(f"Пристрій: {hardware}; PyTorch {torch.__version__}")

labels = list(DATA)
label_ids = {name: index for index, name in enumerate(labels)}
rows = {split: [] for split in ("train", "validation", "test")}
for label, splits in DATA.items():
    for split, block in splits.items():
        rows[split] += [
            {"text": line.strip(), "label": label_ids[label]}
            for line in block.splitlines() if line.strip()
        ]
normalized = [
    " ".join(row["text"].lower().split()) for split in rows.values() for row in split
]
assert len(normalized) == len(set(normalized)) == 168
assert [len(rows[split]) for split in rows] == [112, 24, 32]
for split, examples in rows.items():
    print(split, len(examples), dict(Counter(labels[row["label"]] for row in examples)))
for label, splits in DATA.items():
    print(label, "→", splits["train"].strip().splitlines()[0])

/tmp/ai_systems_lab5_venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Пристрій: CPU; PyTorch 2.9.1+cpu
train 112 {'account': 28, 'connectivity': 28, 'learning_platform': 28, 'printing': 28}
validation 24 {'account': 6, 'connectivity': 6, 'learning_platform': 6, 'printing': 6}
test 32 {'account': 8, 'connectivity': 8, 'learning_platform': 8, 'printing': 8}
account → I forgot my university password and need a reset link.
connectivity → Eduroam keeps disconnecting whenever I sit in the library.
learning_platform → I submitted the assignment in Moodle but it still shows as missing.
printing → The library printer says there is a paper jam but I cannot see any paper.


## Оцінювання до донавчання

Основна метрика — macro-F1. Обчислюємо F1 окремо для кожної черги й даємо кожному класу однакову вагу. Додатково вимірюємо accuracy, частку правильних маршрутів. Базовий рівень завжди повертає найчастішу мітку тренувального набору. Тут усі класи мають однакову частоту, тому заздалегідь використовую першу категорію `account`. Окремо оцінюю випадково ініціалізовану голову Small BERT. Фіксовані вісім тестових рядків буде показано до і після тренування.

In [3]:
CHECKPOINT = "google/bert_uncased_L-2_H-128_A-2"
MAX_LENGTH = 64
BATCH_SIZE = 16
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT, num_labels=len(labels), id2label=dict(enumerate(labels)),
    label2id=label_ids, use_safetensors=True,
).to(device)
print(f"Параметрів моделі: {sum(p.numel() for p in model.parameters()):,}")

def to_batch(samples):
    batch = tokenizer(
        [sample["text"] for sample in samples], padding=True, truncation=True,
        max_length=MAX_LENGTH, return_tensors="pt",
    )
    batch["labels"] = torch.tensor([sample["label"] for sample in samples])
    return batch.to(device)

generator = torch.Generator().manual_seed(SEED)
loaders = {
    split: DataLoader(examples, batch_size=BATCH_SIZE, shuffle=split == "train",
                      generator=generator if split == "train" else None, collate_fn=to_batch)
    for split, examples in rows.items()
}

@torch.inference_mode()
def evaluate(split):
    model.eval()
    predictions = []
    total_loss = 0.0
    for batch in loaders[split]:
        output = model(**batch)
        total_loss += output.loss.item() * batch["labels"].numel()
        predictions.extend(output.logits.argmax(dim=-1).tolist())
    return total_loss / len(rows[split]), predictions

def metrics(truth, guesses):
    correct = sum(actual == guess for actual, guess in zip(truth, guesses))
    scores = []
    for label in range(len(labels)):
        tp = sum(actual == guess == label for actual, guess in zip(truth, guesses))
        false_positive = sum(actual != label and guess == label for actual, guess in zip(truth, guesses))
        false_negative = sum(actual == label and guess != label for actual, guess in zip(truth, guesses))
        denominator = 2 * tp + false_positive + false_negative
        scores.append(2 * tp / denominator if denominator else 0.0)
    return correct / len(truth), sum(scores) / len(scores)

train_frequency = Counter(row["label"] for row in rows["train"])
majority = max(range(len(labels)), key=train_frequency.get)
test_labels = [row["label"] for row in rows["test"]]
majority_predictions = [majority] * len(test_labels)
_, initial_predictions = evaluate("test")
baseline_scores = [metrics(test_labels, pred) for pred in (majority_predictions, initial_predictions)]
print("База (accuracy, macro-F1):", baseline_scores[0])
print("Ненавчена голова (accuracy, macro-F1):", baseline_scores[1])
preview = (0, 6, 8, 10, 16, 21, 24, 31)
for index in preview:
    print(rows["test"][index]["text"], "→", labels[initial_predictions[index]])

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/bert_uncased_L-2_H-128_A-2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Параметрів моделі: 4,386,436
База (accuracy, macro-F1): (0.25, 0.1)
Ненавчена голова (accuracy, macro-F1): (0.1875, 0.15784950343773874)
The portal asks for a second factor that I cannot access. → account
My credentials fail on both Moodle and email before either service opens. → connectivity
Wireless works in the courtyard but drops immediately inside the auditorium. → account
I can open my course from home, but nothing loads on campus Wi-Fi. → account
A course invitation accepted yesterday still hasn't added the class to my dashboard. → account
I can sign in correctly and use email, yet today's lab is hidden in Canvas. → account
The print log shows the file as complete, but the output tray is empty. → account
Only the campus scanner fails to deliver files to my working mailbox. → connectivity


## Донавчання

Оновлюю Base-енкодер і нову голову. AdamW із learning rate `1e-4` дає новій голові змогу навчатись швидко, а кістяк малий, тож можна дозволити оновлення всіх ваг. Батч 16 і максимум 64 токени підходять для коротких звернень і обмеженої пам'яті. До 16 епох. Після кожної епохи вимірюю validation loss і зберігаю найкращий стан у RAM. Якщо validation loss не покращується чотири епохи поспіль, зупиняюся. Тестовий набір не впливає на це рішення.

In [4]:
LEARNING_RATE = 1e-4
MAX_EPOCHS = 16
PATIENCE = 4
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
validation_labels = [row["label"] for row in rows["validation"]]
history = []
best_loss, best_epoch, best_state = float("inf"), 0, None

if device.type == "cuda":
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
start = perf_counter()

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for batch in loaders["train"]:
        optimizer.zero_grad(set_to_none=True)
        loss = model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch["labels"].numel()
    train_loss = total_loss / len(rows["train"])
    validation_loss, validation_predictions = evaluate("validation")
    _, validation_f1 = metrics(validation_labels, validation_predictions)
    history.append((epoch, train_loss, validation_loss, validation_f1))
    if validation_loss < best_loss:
        best_loss, best_epoch = validation_loss, epoch
        best_state = {name: value.cpu().clone() for name, value in model.state_dict().items()}
    elif epoch - best_epoch >= PATIENCE:
        break

if device.type == "cuda":
    torch.cuda.synchronize()
elapsed = perf_counter() - start
peak_gpu = torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else None
model.load_state_dict(best_state)

curve = [
    "| Епоха | Train loss | Validation loss | Validation macro-F1 |",
    "|---:|---:|---:|---:|",
]
for epoch, train_loss, validation_loss, validation_f1 in history:
    curve.append(f"| {epoch} | {train_loss:.4f} | {validation_loss:.4f} | {validation_f1:.3f} |")
display(Markdown("\n".join(curve)))
print(f"Найкраща епоха: {best_epoch}, час: {elapsed:.2f} с")
print(f"Пік VRAM: {peak_gpu:.1f} MiB" if peak_gpu is not None else "Тренування на CPU, VRAM не використовувалась.")

| Епоха | Train loss | Validation loss | Validation macro-F1 |
|---:|---:|---:|---:|
| 1 | 1.3827 | 1.3421 | 0.375 |
| 2 | 1.3456 | 1.3132 | 0.461 |
| 3 | 1.3079 | 1.2750 | 0.733 |
| 4 | 1.2709 | 1.2419 | 0.782 |
| 5 | 1.2306 | 1.1919 | 0.831 |
| 6 | 1.1608 | 1.1553 | 0.672 |
| 7 | 1.1169 | 1.0875 | 0.731 |
| 8 | 1.0491 | 1.0111 | 0.826 |
| 9 | 0.9897 | 0.9603 | 0.826 |
| 10 | 0.9005 | 0.8901 | 0.826 |
| 11 | 0.8475 | 0.8377 | 0.873 |
| 12 | 0.7816 | 0.8018 | 0.826 |
| 13 | 0.7173 | 0.7541 | 0.826 |
| 14 | 0.6645 | 0.7133 | 0.873 |
| 15 | 0.6066 | 0.6681 | 0.873 |
| 16 | 0.5414 | 0.6287 | 0.873 |

Найкраща епоха: 16, час: 5.34 с
Тренування на CPU, VRAM не використовувалась.


In [5]:
_, final_predictions = evaluate("test")
final_accuracy, final_f1 = metrics(test_labels, final_predictions)
headings = (
    "| Варіант | Accuracy | Macro-F1 |\n|---|---:|---:|",
    "| Текст | Еталон | База | До | Після |\n|---|---|---|---|---|",
)
metric_table = [headings[0]]
for name, (accuracy, f1) in zip(
    ("Найчастіша мітка", "Ненавчена голова", "Після донавчання"),
    [*baseline_scores, (final_accuracy, final_f1)],
):
    metric_table.append(f"| {name} | {accuracy:.3f} | {f1:.3f} |")
display(Markdown("\n".join(metric_table)))

example_table = [headings[1]]
for index in preview:
    text = rows["test"][index]["text"]
    gold, base, initial, final = [
        labels[values[index]]
        for values in (test_labels, majority_predictions, initial_predictions, final_predictions)
    ]
    example_table.append(f"| {text} | {gold} | {base} | {initial} | {final} |")
display(Markdown("\n".join(example_table)))

errors = [
    (row["text"], labels[row["label"]], labels[predicted])
    for row, predicted in zip(rows["test"], final_predictions)
    if row["label"] != predicted
]
print(f"Помилок після донавчання: {len(errors)} із {len(test_labels)}.")
for text, gold, prediction in errors:
    print(f"{gold} → {prediction}: {text}")

| Варіант | Accuracy | Macro-F1 |
|---|---:|---:|
| Найчастіша мітка | 0.250 | 0.100 |
| Ненавчена голова | 0.188 | 0.158 |
| Після донавчання | 0.906 | 0.905 |

| Текст | Еталон | База | До | Після |
|---|---|---|---|---|
| The portal asks for a second factor that I cannot access. | account | account | account | account |
| My credentials fail on both Moodle and email before either service opens. | account | account | connectivity | account |
| Wireless works in the courtyard but drops immediately inside the auditorium. | connectivity | account | account | connectivity |
| I can open my course from home, but nothing loads on campus Wi-Fi. | connectivity | account | account | connectivity |
| A course invitation accepted yesterday still hasn't added the class to my dashboard. | learning_platform | account | account | learning_platform |
| I can sign in correctly and use email, yet today's lab is hidden in Canvas. | learning_platform | account | account | account |
| The print log shows the file as complete, but the output tray is empty. | printing | account | account | printing |
| Only the campus scanner fails to deliver files to my working mailbox. | printing | account | connectivity | account |

Помилок після донавчання: 3 із 32.
learning_platform → account: I can sign in correctly and use email, yet today's lab is hidden in Canvas.
printing → connectivity: I can reach the internet, but the wireless printer won't accept any jobs.
printing → account: Only the campus scanner fails to deliver files to my working mailbox.


## 1.6. Мій звіт

- **Задача і чому потрібне донавчання.** Маршрутизація англомовних звернень у чотири локальні черги університетської IT-підтримки. Донавчений класифікатор застосовує задану розмітку й повертає одну мітку за прямий прохід.
- **Модель і дані.** Google Small BERT з новою головою. Власні синтетичні приклади під CC0, 112 для тренування, 24 для валідації, 32 для тесту.
- **Метрика й результат до донавчання.** Основна метрика macro-F1, додаткова accuracy. Найчастіша мітка дала **0,100 macro-F1 та 25,0% accuracy**. Випадкова голова дала **0,158 macro-F1 та 18,8% accuracy**.
- **Метод і гіперпараметри.** Повне донавчання, AdamW, learning rate 1e-4, батч 16, максимум 16 епох. Найкращий стан вибрано за validation loss на 16-й епосі. Зафіксовано **5,34 с** тренування та валідації на CPU, без використання VRAM. Train loss знизився з 1,3827 до 0,5414, validation loss — з 1,3421 до 0,6287. За validation loss погіршення наприкінці не спостерігалось.
- **Результат після донавчання.** **0,905 macro-F1 та 90,6% accuracy** на тих самих 32 зверненнях. Приріст macro-F1 проти бази становить 0,805.
- **Аналіз помилок.** Три помилки виникли на межі категорій. Імовірно, модель надавала забагато ваги окремим словам. Звернення про прихований матеріал у Canvas містило згадки про вхід і пошту, звернення про принтер — *wireless* та *internet*, а звернення про сканер — згадку про поштову скриньку. Ці слова узгоджуються з помилковими передбаченнями. Можливе поліпшення — додати до train нові контрастні пари, де ці слова лишаються, а причина звернення змінюється. Поточні тестові приклади зберігаю для оцінювання.
- **Висновок.** Донавчання суттєво покращило маршрутизацію в цьому синтетичному тесті. Перед робочим використанням потрібен окремий тест на реальних анонімізованих зверненнях, особливо з кількома потенційними категоріями.

Нижче наведено показники поточного запуску для відтворення експерименту.

In [6]:
improvement = final_f1 - baseline_scores[0][1]
report_lines = [
    f"**До донавчання.** Accuracy / macro-F1 найчастішої мітки "
    f"{baseline_scores[0][0]:.3f} / {baseline_scores[0][1]:.3f}; випадкової голови "
    f"{baseline_scores[1][0]:.3f} / {baseline_scores[1][1]:.3f}.",
    f"**Після донавчання.** Accuracy {final_accuracy:.3f}, macro-F1 {final_f1:.3f}; "
    f"зміна macro-F1 від бази {improvement:+.3f}. Найкраща епоха {best_epoch} "
    f"з {len(history)}, час {elapsed:.2f} с на {hardware}.",
    f"Пік виділеної пам'яті GPU {peak_gpu:.1f} MiB." if peak_gpu is not None
    else "Експеримент виконано на CPU, тому пік GPU-пам'яті не вимірювався.",
    f"**Аналіз.** На тесті {len(errors)} помилок; точні приклади надруковано вище.",
]
if best_epoch < len(history) and history[-1][1] < history[best_epoch - 1][1]:
    report_lines.append("Після мінімуму validation loss тренувальний loss ще знижувався. Для тесту взято раніший стан моделі.")
display(Markdown("\n\n".join(report_lines)))

**До донавчання.** Accuracy / macro-F1 найчастішої мітки 0.250 / 0.100; випадкової голови 0.188 / 0.158.

**Після донавчання.** Accuracy 0.906, macro-F1 0.905; зміна macro-F1 від бази +0.805. Найкраща епоха 16 з 16, час 5.34 с на CPU.

Експеримент виконано на CPU, тому пік GPU-пам'яті не вимірювався.

**Аналіз.** На тесті 3 помилок; точні приклади надруковано вище.